# Progression Model
Explore the possibilities of a progression model that predicts a lifter's next total given their previous total, time between meets, previous DOTS score, change in bodyweight.

In [1]:
import os
import sys

import pandas as pd
from sqlalchemy import create_engine

engine = create_engine('sqlite:///../data/powerlifting.db')

# Shape
pd.read_sql("SELECT COUNT(*) as total_rows FROM meets", engine)

,total_rows
0,3907789


In [2]:
# Check GL Point completeness and Date format
pd.read_sql("""
SELECT
    COUNT(*) as total,
    COUNT(Goodlift) as has_GL,
    COUNT(Date) as has_date,
    MIN(Date) as earliest,
    MAX(Date) as latest
FROM meets
WHERE Equipment = 'Raw'
    AND Event = 'SBD'
    AND TotalKg IS NOT NULL
    AND Age IS NOT NULL
    AND Sex IN ('M', 'F')
    AND Place NOT IN ('DQ', 'NH', 'NS', 'NQ')
    AND Age >= 14
    AND Age <= 80
""", engine)

,total,has_GL,has_date,earliest,latest
0,906983,906441,906983,1965-09-04,2026-04-19


In [3]:
# Check how many lifters have multiple meets
pd.read_sql("""
    SELECT
        meet_count,
        COUNT(*) as num_lifters
    FROM (
        SELECT Name, COUNT(*) as meet_count
        FROM meets
        WHERE Equipment = 'Raw'
            AND Event = 'SBD'
            AND TotalKg IS NOT NULL
            AND Age IS NOT NULL
            AND BodyweightKg IS NOT NULL
            AND Sex IN ('M', 'F')
            AND Place NOT IN ('DQ', 'NH', 'NS', 'NQ')
            AND Age >= 14
            AND Age <= 80
        GROUP BY Name
    ) 
    GROUP BY meet_count
    ORDER BY meet_count
""", engine)

,meet_count,num_lifters
0,1,144356
1,2,67599
2,3,35961
3,4,22816
4,5,14835
5,6,10122
6,7,6957
7,8,5019
8,9,3583
9,10,2731


In [2]:
os.chdir('/Users/lawrence/pl-predictor')
sys.path.append('/Users/lawrence/pl-predictor')

from src.progression_model import train

model, le, X_test, y_test, preds, transitions = train()

Loading raw data...
Raw rows: 899,994
Engineering features...
Transition rows: 490,667
          prev_total        prev_gl  prev_bodyweight  next_bodyweight  \
count  490667.000000  490667.000000    490667.000000    490667.000000   
mean      476.867638      72.448273        82.959373        83.803555   
std       154.587974      13.792507        20.883553        21.182192   
min        47.500000       8.550000        35.000000        35.300000   
25%       345.000000      63.470000        67.300000        67.900000   
50%       480.000000      72.790000        81.150000        81.600000   
75%       592.500000      81.770000        94.600000        96.500000   
max      1152.500000     137.670000       234.000000       233.300000   

           bw_change    age_at_next  months_between    meet_number  \
count  490667.000000  490667.000000   490667.000000  490667.000000   
mean        0.844183      29.350561        8.736822       3.426024   
std         4.998582      11.716107        8.